# Etapa 5: Engenharia de Features

**Projeto Integrador (P15):** Previsão de Alta Demanda de Produtos da Bioeconomia Amazônica
Disciplina Banco de Dados e Engenharia de Dados para IA, Prof. Adolfo Colares, [UNIFAP – Universidade Federal do Amapá](https://www.unifap.br/)
Autor: Leonam Souza dos Santos Azevedo

**Objetivo:** transformar o modelo estrela em uma base de modelagem, com uma linha por município, produto e mês, contendo as features e o alvo `alta_demanda`.

**Princípio central: nada de vazamento de dados.** Toda feature usa apenas informação disponível **até o mês anterior** ao previsto (t-1). Assim o modelo pode ser usado de verdade no início de cada mês, e as métricas da Etapa 6 refletem o desempenho real. As features implementam as decisões da análise exploratória (`01_eda.ipynb`, seção 10).

O cálculo está em `src/features.py`; o script `python -m src.build_features` gera a base e a grava no PostgreSQL (schema `ml`). Este notebook documenta, testa e analisa essa base.

**Roteiro**

1. Construção da base de modelagem
2. Dicionário de features
3. Exemplo: uma série ao longo do tempo
4. Teste automático de vazamento de dados
5. Valores ausentes e tratamento
6. Comportamento das features por classe
7. Relevância das features
8. Divisão temporal entre treino e teste
9. Conferência com a base gravada
10. Conclusões

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

from src import config
from src.features import (
    ALVO, FEATURES, FEATURES_CATEGORICAS, FEATURES_NUMERICAS,
    adicionar_alvo, completar_calendario, construir_features, dicionario_features,
)
from src.leitura import carregar_dw, montar_base_vendas

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
pd.set_option("display.max_colwidth", None)

CORES_CLASSE = {0: "#9aa5b1", 1: "#2f6f4e"}
FIGURAS = config.REPORTS_DIR / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

def salvar(fig, nome):
    fig.savefig(FIGURAS / f"{nome}.png", bbox_inches="tight", dpi=150)

## 1. Construção da base de modelagem

In [ ]:
dw, origem = carregar_dw()
print(f"Fonte dos dados: {origem}")

base = montar_base_vendas(dw)
features = construir_features(base)

series = base.groupby(["cod_ibge", "id_produto"]).ngroups
possiveis = series * 24  # jan/2023 a dez/2024
print(f"Séries município + produto: {series}")
print(f"Registros possíveis com alvo (2023 e 2024): {possiveis:,}")
print(f"Registros na base de modelagem: {len(features):,} ({possiveis - len(features)} sem alvo)")
print(f"Features: {len(FEATURES)} ({len(FEATURES_NUMERICAS)} numéricas e {len(FEATURES_CATEGORICAS)} categóricas)")
features.head()

Os meses de 2022 entram apenas como histórico: o alvo exige os 12 meses anteriores completos, então começa em janeiro de 2023. Os poucos registros possíveis sem alvo pertencem à série que perdeu um mês no ETL: enquanto esse mês ausente estiver dentro da janela de 12 meses, a média histórica não é calculada, em vez de ser estimada com dados incompletos.

## 2. Dicionário de features

In [ ]:
dicionario = dicionario_features()
dicionario

As 16 features se dividem em quatro grupos, cada um ligado a um achado da EDA:

- **Memória recente:** a persistência entre meses consecutivos foi o sinal mais forte da EDA.
- **Sazonalidade:** cada produto tem o seu calendário; o índice sazonal histórico já vem calculado por produto, e o seno e o cosseno do mês permitem que os modelos de árvore combinem mês e produto.
- **Preço:** o preço se move no sentido oposto à quantidade e antecipa a fase da safra.
- **Contexto:** população, produção extrativa, produto e UF não preveem o alvo sozinhos, mas ajudam o modelo a ajustar os demais sinais.

As variáveis de volume entram como **log da razão** sobre a média histórica: o valor 0 significa "igual à média"; positivo, acima; negativo, abaixo. Isso coloca municípios de tamanhos muito diferentes na mesma escala.

## 3. Exemplo: uma série ao longo do tempo

In [ ]:
completo = adicionar_alvo(completar_calendario(base))
completo["data"] = pd.to_datetime(completo["id_tempo"].astype(str), format="%Y%m")

cod_exemplo = base.loc[base["populacao_2022"].idxmax(), "cod_ibge"]
exemplo = completo[(completo["cod_ibge"] == cod_exemplo) & (completo["chave"] == "acai")]
nome_exemplo = f"{exemplo['municipio'].iat[0]}/{exemplo['uf'].iat[0]}"

fig, eixo = plt.subplots(figsize=(13, 4.5))
eixo.plot(exemplo["data"], exemplo["quantidade"] / 1000, color="#5b2a6e", label="quantidade vendida")
eixo.plot(exemplo["data"], exemplo["media_12m_anteriores"] / 1000, color="black", linestyle="--",
          label="média dos 12 meses anteriores")
for classe, rotulo in [(1, "alta demanda"), (0, "baixa demanda")]:
    pontos = exemplo[exemplo["alta_demanda"] == classe]
    eixo.scatter(pontos["data"], pontos["quantidade"] / 1000, color=CORES_CLASSE[classe], s=45, zorder=3, label=rotulo)
eixo.axvspan(exemplo["data"].min(), pd.Timestamp("2022-12-31"), color="gray", alpha=0.08)
eixo.text(pd.Timestamp("2022-02-01"), eixo.get_ylim()[1] * 0.92, "2022: só histórico", color="gray")
eixo.set_ylabel("mil kg")
eixo.set_title(f"Açaí em {nome_exemplo}: como o alvo é definido")
eixo.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=4, frameon=False)
salvar(fig, "10_exemplo_alvo")
plt.show()

colunas = ["id_tempo", ALVO, "log_razao_lag1", "log_razao_media3", "alta_mes_anterior",
           "indice_sazonal_hist", "log_preco_rel_lag1"]
features[(features["cod_ibge"] == cod_exemplo) & (features["chave"] == "acai")][colunas].head(8)

O alvo vale 1 quando a linha roxa passa acima da linha tracejada, que acompanha o nível recente do município. Na tabela, os meses de alta demanda tendem a ter `log_razao_lag1` positivo (o mês anterior já estava acima da média) e índice sazonal acima de 1 (época de safra).

## 4. Teste automático de vazamento de dados

Para provar que nenhuma feature olha para o próprio mês ou para o futuro, o teste altera artificialmente as vendas de um mês M (quantidade e receita multiplicadas por 10, preço multiplicado por 3) e recalcula todas as features.

- Se alguma feature do **mês M ou de meses anteriores** mudar, há vazamento.
- As features dos **meses seguintes** devem mudar, porque passam a enxergar M como passado. Isso prova que o teste é sensível.

In [ ]:
CHAVE = ["cod_ibge", "id_produto", "id_tempo"]
original = features.set_index(CHAVE)[FEATURES_NUMERICAS].astype(float)

def testar_vazamento(mes_alterado: int) -> dict:
    alterada = base.copy()
    no_mes = alterada["id_tempo"] == mes_alterado
    alterada.loc[no_mes, ["quantidade", "receita"]] *= 10
    alterada.loc[no_mes, "preco_unitario"] *= 3
    nova = construir_features(alterada).set_index(CHAVE)[FEATURES_NUMERICAS].astype(float).reindex(original.index)

    mudou = ~np.isclose(original.to_numpy(), nova.to_numpy(), equal_nan=True)
    ate_m = (original.index.get_level_values("id_tempo") <= mes_alterado)
    return {
        "mês alterado": mes_alterado,
        "valores alterados até o mês M (deve ser 0)": int(mudou[ate_m].sum()),
        "valores alterados depois de M (deve ser > 0)": int(mudou[~ate_m].sum()),
    }

resultado = pd.DataFrame([testar_vazamento(m) for m in (202306, 202405)])
display(resultado)

assert (resultado["valores alterados até o mês M (deve ser 0)"] == 0).all(), "Vazamento detectado!"
assert (resultado["valores alterados depois de M (deve ser > 0)"] > 0).all(), "Teste insensível!"
print("Nenhum vazamento: as features de um mês dependem só de meses anteriores.")

O teste passou: alterar as vendas de um mês não muda nenhuma feature daquele mês nem dos anteriores, e muda as dos meses seguintes. Esse teste roda automaticamente sempre que o notebook é executado; se uma alteração futura no código introduzir vazamento, ele falha.

## 5. Valores ausentes e tratamento

In [ ]:
ausentes = features.groupby("conjunto")[FEATURES].apply(lambda d: d.isna().mean() * 100).T
ausentes = ausentes[(ausentes > 0).any(axis=1)].round(1)
ausentes.columns = [f"% ausente ({c})" for c in ausentes.columns]
ausentes

| Feature | Por que fica vazia | Tratamento na Etapa 6 |
|---------|--------------------|-----------------------|
| `alta_mes_anterior` | Em janeiro de 2023, porque dezembro de 2022 não tem alvo | Regressão logística: imputação pela mediana com indicador de ausência. Modelos de árvore (HistGradientBoosting): tratamento nativo de valores ausentes |
| `log_producao_ano_anterior` | O IBGE não divulgou a produção (símbolo `...`) | Mesmo tratamento; a feature `producao_indisponivel` já registra o motivo |
| Features de razão | Mês ausente dentro da janela da série que perdeu um registro no ETL | Mesmo tratamento |

A imputação é ajustada **apenas no treino**, dentro do pipeline do modelo, para que o conjunto de teste não influencie o pré-processamento.

## 6. Comportamento das features por classe

In [ ]:
treino = features[features["conjunto"] == "treino"]
destaque = ["log_razao_lag1", "log_razao_media3", "log_preco_rel_lag1", "indice_sazonal_hist"]

fig, eixos = plt.subplots(1, 4, figsize=(17, 4))
for eixo, nome in zip(eixos, destaque):
    sns.boxplot(data=treino, x=ALVO, y=nome, hue=ALVO, palette=CORES_CLASSE, showfliers=False, legend=False, ax=eixo)
    eixo.set_xticks([0, 1])
    eixo.set_xticklabels(["baixa", "alta"])
    eixo.set_xlabel("demanda no mês")
    eixo.set_title(nome)
fig.suptitle("Distribuição das principais features por classe (treino, 2023)", y=1.03)
salvar(fig, "11_features_por_classe")
plt.show()

treino.groupby(ALVO)[destaque].median().rename(index={0: "mediana na baixa demanda", 1: "mediana na alta demanda"})

As features de memória recente e de sazonalidade separam bem as classes: nos meses de alta demanda, o mês anterior já tende a estar acima da média histórica e o produto está em época de safra. O preço relativo do mês anterior se comporta no sentido oposto, menor na alta demanda, como a EDA indicou.

## 7. Relevância das features

In [ ]:
# Informação mútua: mede dependência (inclusive não linear) entre cada feature e o alvo.
# Calculada só no treino, para não usar informação do teste.
X_mi = treino[FEATURES_NUMERICAS].fillna(treino[FEATURES_NUMERICAS].median())
X_mi = X_mi.join(pd.get_dummies(treino[FEATURES_CATEGORICAS], dtype=float))
binarias = [X_mi[col].nunique() <= 2 for col in X_mi.columns]

relevancia = pd.DataFrame(
    {
        "informacao_mutua": mutual_info_classif(X_mi, treino[ALVO], discrete_features=binarias, random_state=42),
        "correlacao_com_alvo": X_mi.corrwith(treino[ALVO].astype(float)),
    },
    index=X_mi.columns,
).sort_values("informacao_mutua", ascending=False)

fig, eixo = plt.subplots(figsize=(9, 6))
eixo.barh(relevancia.index[::-1], relevancia["informacao_mutua"][::-1], color="#2f6f4e")
eixo.set_xlabel("informação mútua com o alvo")
eixo.set_title("Relevância das features (treino, 2023)")
salvar(fig, "12_relevancia_features")
plt.show()

relevancia

As features de **memória recente** e de **sazonalidade** concentram a maior parte da informação sobre o alvo. As de contexto (população, produção, UF) ficam com relevância baixa quando avaliadas sozinhas, o que confirma a EDA; elas são mantidas porque modelos de árvore podem usá-las em interações (por exemplo, a mesma razão do mês anterior pode significar coisas diferentes para produtos diferentes). A seleção final será avaliada pela importância das features no modelo treinado, na Etapa 6.

## 8. Divisão temporal entre treino e teste

In [ ]:
divisao = features.groupby(["conjunto", "nome"])[ALVO].agg(registros="size", taxa_alta_demanda="mean").unstack("conjunto")
display(divisao)

taxa_mensal = features.groupby(["id_tempo", "nome"])[ALVO].mean().unstack("nome") * 100
taxa_mensal.index = pd.to_datetime(taxa_mensal.index.astype(str), format="%Y%m")

fig, eixo = plt.subplots(figsize=(13, 4))
cores_produto = {"Açaí": "#5b2a6e", "Castanha-do-pará": "#8c5a2b", "Óleo de copaíba": "#c9a227"}
for produto in taxa_mensal.columns:
    eixo.plot(taxa_mensal.index, taxa_mensal[produto], marker="o", markersize=3, color=cores_produto[produto], label=produto)
eixo.axvline(pd.Timestamp("2023-12-16"), color="black", linestyle="--")
eixo.text(pd.Timestamp("2023-05-01"), 95, "TREINO (2023)", fontweight="bold")
eixo.text(pd.Timestamp("2024-05-01"), 95, "TESTE (2024)", fontweight="bold")
eixo.set_ylim(0, 105)
eixo.set_ylabel("% de registros com alta demanda")
eixo.set_title("Taxa mensal de alta demanda e divisão temporal")
eixo.legend(loc="lower left", ncol=3)
salvar(fig, "13_divisao_temporal")
plt.show()

A divisão é **temporal**: o modelo aprende com 2023 e é avaliado em 2024, sem embaralhar os registros. Isso reproduz o uso real (prever meses que ainda não aconteceram) e impede que o modelo seja avaliado em meses anteriores aos que viu no treino. Os dois conjuntos têm tamanho e taxa de alta demanda parecidos, e cada um cobre um ano completo, com todas as fases da safra.

## 9. Conferência com a base gravada

In [ ]:
try:
    from sqlalchemy import text
    from src.db import criar_engine

    engine = criar_engine()
    with engine.connect() as conn:
        gravada = pd.read_sql(text("SELECT * FROM ml.features_demanda"), conn)
    engine.dispose()
    fonte_gravada = "PostgreSQL (ml.features_demanda)"
except Exception as erro:
    gravada = pd.read_csv(config.DATA_PROCESSED / "features_modelo.csv")
    fonte_gravada = f"CSV (data/processed/features_modelo.csv); banco indisponível: {type(erro).__name__}"

chave = ["cod_ibge", "id_produto", "id_tempo"]
comparacao = features.set_index(chave)[FEATURES_NUMERICAS + [ALVO]].astype(float).sort_index()
gravada_ordenada = gravada.set_index(chave)[FEATURES_NUMERICAS + [ALVO]].astype(float).sort_index()

print(f"Base gravada lida de: {fonte_gravada}")
print(f"Linhas: notebook = {len(comparacao):,} | gravada = {len(gravada_ordenada):,}")
print("Conteúdo idêntico:", comparacao.shape == gravada_ordenada.shape
      and np.allclose(comparacao.to_numpy(), gravada_ordenada.to_numpy(), equal_nan=True))

A base usada na Etapa 6 é a gravada pelo pipeline (`python -m src.build_features`), e a conferência acima garante que ela é idêntica à analisada neste notebook.

## 10. Conclusões

- A base de modelagem tem **uma linha por município, produto e mês** de 2023 e 2024, com **16 features** em quatro grupos (memória recente, sazonalidade, preço e contexto) e o alvo `alta_demanda`.
- O **teste automático de vazamento** comprova que todas as features usam apenas informação anterior ao mês previsto.
- Os valores ausentes têm causa conhecida e serão tratados dentro do pipeline do modelo, ajustado só no treino.
- As features de **memória recente** e **sazonalidade** são as mais informativas; as de contexto entram para permitir interações.
- A **divisão temporal** (treino em 2023, teste em 2024) reproduz o uso real do modelo.

**Próximo passo (Etapa 6):** treinar e comparar uma regressão logística (referência interpretável), uma Random Forest e um modelo de gradient boosting, medindo Accuracy e F1 no teste de 2024 contra uma regra ingênua ("repetir o resultado do mês anterior").